In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS electronics_retailer_clg.silver;

In [0]:
from pyspark.sql.functions import col, trim

df = spark.table("electronics_retailer_clg.bronze.stores")


df = df.toDF(*[c.lower().replace(" ", "_") for c in df.columns])


for c in df.columns:
    df = df.withColumn(c, trim(col(c)))



df = df.withColumn("storekey", col("storekey").cast("int"))


df = df.fillna({
    "country": "unknown"
})


df = df.select(
    "storekey",
    "country"
)


df = df.dropDuplicates(["storekey"])



display(df)
df.printSchema()


df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("electronics_retailer_clg.silver.stores")

print("Store cleaned successfully")